# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We review all available `RecordSet` entities and the fields/columns within each. **All identifiers use their `@id` as required.**

In [ ]:
from collections.abc import Iterable

print("Available record sets and their fields:\n")
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}  (name: {getattr(record_set, 'name', 'N/A')})")
    print("  Fields/columns in this record set:")
    for field in getattr(record_set, 'fields', []):
        field_id = getattr(field, 'id', None)
        field_name = getattr(field, 'name', 'N/A')
        print(f"    - Field @id: {field_id}  (name: {field_name})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

First, pick the main record set (usually the one with the core table of interest). As there may be multiple, we choose the first here as an example.

In [ ]:
# List record set @ids
record_set_ids = [rs.id for rs in metadata.record_sets]
print("Record set ids: ", record_set_ids)
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet {record_set_id} with shape {df.shape}")

# For demonstration, select the first record set id
main_record_set_id = record_set_ids[0]
print("\nColumns in main DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())

dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**All columns are referenced by their `@id`** as required. If the columns contain special/exotic IDs, you may want to print those first and select a meaningful numeric field.

For this demonstration we'll select a numeric field if available (e.g., coverage or MW).

In [ ]:
# List all columns with their @id (column name) and types
df = dataframes[main_record_set_id]
print("All available columns (use `@id` for processing):\n", df.columns.tolist())

# Try to select a numeric field for EDA
import numpy as np

# Heuristics: look for likely numeric columns ('coverage', 'MW', or similar)
numeric_candidates = [c for c in df.columns if any(k in c.lower() for k in ['coverage', 'mw', 'weight', 'count', 'abundance', 'number'])]
print("Numeric column candidates: ", numeric_candidates)

# If found, pick the first one; otherwise pick any float/integer column
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    # Try auto-detect float/integer columns
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
        else:
            numeric_field_id = None

print("Using numeric field for analysis (must be @id): ", numeric_field_id)

# Drop NAs, make numeric if possible (for filtering)
df_clean = df.copy()
df_clean[numeric_field_id] = pd.to_numeric(df_clean[numeric_field_id], errors='coerce')
threshold = df_clean[numeric_field_id].median()  # Example: use median as a threshold
filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > median ({threshold}): Shape: {filtered_df.shape}")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field (Z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a field (prefer a categorical, e.g. sample/condition/description)
group_field_candidates = [c for c in df.columns if c != numeric_field_id and any(g in c.lower() for g in ["sample", "group", "type", "condition", "description"])]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
    print(f"\nGrouping by field (as @id): {group_field_id}\n")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(grouped_df.head())
else:
    print("\nNo suitable categorical field for grouping found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We'll make some basic plots for the selected numeric field.

You can extend this section with your preferred visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df_clean[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

if 'group_field_id' in locals():
    plt.figure(figsize=(8,6))
    sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded and explored the Mass Spectrometry dataset (referenced entirely using Croissant `@id`s).
- Key numeric field(s) were analyzed and visualized.
- More advanced domain-specific analytics can be pursued using this reproducible workflow.